In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from sklearn.model_selection import train_test_split
import numpy as np
from xgboost import XGBClassifier
from tree.mini_xgboost_classifier import MiniXGBoostClassifier
from sklearn.metrics import accuracy_score
from tree.vectorized_single_hist_tree_boost import VectorizedSingleHistTreeBoost
from sklearn.datasets import make_classification
import shap
import matplotlib.pyplot as plt
import pandas as pd
from typing import Tuple
import polars as pl

/home/sakane/Documents/projects/python/mini_xgshap/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pl.read_parquet("../datasets/BRFSS/out.parquet")

In [3]:
df.head()

DIABETES,SEX,INCOME,HIGHBP,HIGHCHOL,DRINK,INSURANCE,AGE,RACE,MARITAL,EDUCATION,EMPLOYMENT,BMI,HEALTH,PHYSICAL,MENTAL,SMOKER,EXERCISE,TOTAL_COMORBIDITY,TOTAL_DISABILITY
u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
1,2,null,2,1,1,null,6,1,1,3,7,4,1,1,1,4,2,0,1
3,2,null,2,2,1,null,6,1,2,3,7,3,1,1,1,4,1,2,1
3,2,1,2,2,1,null,6,2,3,2,7,2,2,2,2,3,1,2,3
3,2,null,1,1,1,null,6,1,1,3,7,3,1,2,1,4,1,4,1
1,2,5,2,1,1,null,6,1,3,3,8,3,2,1,1,4,1,3,3


In [5]:
df.describe()

statistic,DIABETES,SEX,INCOME,HIGHBP,HIGHCHOL,DRINK,INSURANCE,AGE,RACE,MARITAL,EDUCATION,EMPLOYMENT,BMI,HEALTH,PHYSICAL,MENTAL,SMOKER,EXERCISE,TOTAL_COMORBIDITY,TOTAL_DISABILITY
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.722563e6,1.722287e6,1.390882e6,1.71652e6,1.503255e6,1.598624e6,1.047896e6,1.722563e6,1.685263e6,1.707283e6,1.714693e6,1.697147e6,1.565672e6,1.718134e6,1.683266e6,1.691241e6,1.638215e6,1.665077e6,1.722563e6,1.722563e6
"""null_count""",0.0,276.0,331681.0,6043.0,219308.0,123939.0,674667.0,0.0,37300.0,15280.0,7870.0,25416.0,156891.0,4429.0,39297.0,31322.0,84348.0,57486.0,0.0,0.0
"""mean""",2.748881,1.538847,3.93255,1.404142,1.400504,1.05908,1.102198,4.414377,2.010756,2.311832,3.005799,3.849306,2.988675,1.186186,1.5028,1.481601,3.363602,1.25932,1.067932,0.523397
"""std""",0.710784,0.498489,1.370112,0.490725,0.490001,0.235775,0.302909,1.603203,2.203767,1.671754,0.960107,2.852659,0.832428,0.389257,0.72017,0.705548,0.9331,0.438262,1.233493,0.992781
"""min""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""25%""",3.0,1.0,3.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,3.0,1.0,0.0,0.0
"""50%""",3.0,2.0,5.0,1.0,1.0,1.0,1.0,5.0,1.0,1.0,3.0,2.0,3.0,1.0,1.0,1.0,4.0,1.0,1.0,0.0
"""75%""",3.0,2.0,5.0,2.0,2.0,1.0,1.0,6.0,1.0,3.0,4.0,7.0,4.0,1.0,2.0,2.0,4.0,2.0,2.0,1.0
"""max""",4.0,2.0,5.0,2.0,2.0,2.0,2.0,6.0,8.0,6.0,4.0,8.0,4.0,2.0,3.0,3.0,4.0,2.0,9.0,6.0


In [ ]:
max_depth = 5
learning_rate = 0.1
n_estimators = 300
max_bin = 30
tree_method = 'hist'
gamma = 0
lb = 1